# conveyor-perception — Live Colab Demo

**Run all cells** (`Runtime → Run all`) to see the full industrial perception stack in action:

1. Train YOLO26s on the recycling dataset (~10-15 min on Colab T4)
2. Run the multi-task pipeline (detect → track → drift → triage) on a sample image
3. See the L1 triage queue + drift signals + maintenance hints in one view
4. Run a robustness test against MRF conditions
5. Get a shift dashboard snapshot

**No setup needed** — the notebook installs everything, downloads the dataset, and runs the demo. Total runtime: ~20 minutes.

**What this demonstrates** (mapped to the EverestLabs JD):
- **Real-time detection** (JD #1): YOLO26 NMS-free, 2.5ms T4
- **Predictive maintenance** (JD #2): drift signals → actionable hints
- **Multi-model pipeline** (JD #3): detection + tracking + drift + triage in one frame
- **ROS 2 integration** (JD #4): code path exists; this notebook uses Python directly
- **Robustness** (JD #5): 13 MRF-condition augmentations, broken/degraded/ok report
- **Monitoring** (JD #6): shift dashboard with retrain recommendation
- **L1 triage** (bonus): MCP-style 5-tool surface for the ROC agent

---

## Cell 1: Install dependencies (~60s)

In [ ]:
!pip install -q ultralytics==8.4.121 opencv-python==4.11.0.86 supervision==0.30.0 roboflow==1.4.1 fastmcp==3.4.7 pydantic==2.13.4 numpy==1.26.4
!pip install -q python-dotenv>=1.1.0
print("✓ Dependencies installed")

## Cell 2: Clone the repo

In [ ]:
!git clone https://github.com/roniejosephv-star/conveyor-perception.git 2>/dev/null || (cd conveyor-perception && git pull)
%cd conveyor-perception
print("✓ Repo ready")

## Cell 3: Download the recycling dataset (~30s)

The dataset is **zkf624/-recycling v3** (CC BY 4.0): 2,404 images, 4 classes (Glass, metal, plastic, vinyl). Roboflow Universe, 99.5% pre-trained mAP@50.

In [ ]:
# Set up the API key for Roboflow (you can use a free public key or skip if you have one cached)
import os

if not os.path.exists('.env'):
    # The repo's .gitignore excludes .env; we use a public read-only key here for the demo.
    # Get your own at https://app.roboflow.com → Settings → API Keys
    with open('.env', 'w') as f:
        f.write('ROBOFLOW_API_KEY=qogO5hAuLgUUYMbNT6W3\n')
    print("Wrote demo .env (use your own key for real work)")

!python scripts/download_dataset.py 2>&1 | tail -10

## Cell 4: Train YOLO26s on the recycling data (~10-15 min on T4)

In [ ]:
!python scripts/train_yolo26.py \
    --epochs 30 \
    --imgsz 640 \
    --batch 32 \
    --device 0 \
    --data-yaml data/raw/recycling_v3/data.yaml \
    2>&1 | tail -20

## Cell 5: Run the multi-task pipeline on a sample image

Wires together **Detector → Tracker → DriftMonitor → L1TriageAgent → MaintenanceAdvisor**.

In [ ]:
import sys

sys.path.insert(0, '.')

import cv2

from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.monitoring.dashboard import MonitoringDashboard
from conveyor_perception.multitask.pipeline import MultitaskPipeline
from conveyor_perception.perception.detector import Detector
from conveyor_perception.predictive_maintenance.advisor import (
    MaintenanceAdvisor,
)
from conveyor_perception.triage.agent import L1TriageAgent

# Build all the components
print("Building pipeline components...")
detector = Detector(
    model_path='models/yolo26s_recyclable.onnx',
    class_names=['Glass', 'metal', 'plastic', 'vinyl'],
    conf_threshold=0.25,
)
tracker = TrackingPipeline()
drift = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
triage = L1TriageAgent()
advisor = MaintenanceAdvisor()
dashboard = MonitoringDashboard()
pipeline = MultitaskPipeline(detector, tracker, drift, triage)

# Download a sample image (or use the one from the repo)
import os
import urllib.request

os.makedirs('data/sample', exist_ok=True)
if not os.path.exists('data/sample/bus.jpg'):
    urllib.request.urlretrieve(
        'https://ultralytics.com/images/bus.jpg',
        'data/sample/bus.jpg'
    )
image = cv2.imread('data/sample/bus.jpg')
print(f"Image shape: {image.shape}")

# Run 60 frames to accumulate drift signals
for i in range(60):
    result = pipeline.step(image)
    dashboard.record_frame(result)

print(f"\n✓ Pipeline ran 60 frames, {sum(len(r.detections) for r in [result])} detections in last frame")

## Cell 6: Inspect the L1 triage queue + alerts

In [ ]:
print("=== Triage queue (most recent 10 alerts) ===\n")
for alert in triage.get_pending(limit=10):
    print(f"  [{alert.severity.upper():9s}] {alert.class_name:10s} "
          f"conf={alert.confidence:.2f} reason='{alert.metadata.get('reason', '')[:50]}'")

print("\n=== Triage stats ===")
stats = triage.get_stats()
for k, v in stats.to_dict().items():
    print(f"  {k}: {v}")

## Cell 7: Robustness test — the JD's 'chaotic environments' requirement

In [ ]:
from conveyor_perception.robustness import RobustnessTestSuite

print("Running robustness suite on the same image...\n")
suite = RobustnessTestSuite(detector, image)
report = suite.run()
print(report.to_markdown())

## Cell 8: Shift dashboard — the supervisor's 8am report

In [ ]:
import json

shift_report = dashboard.shift_report()
print(json.dumps(shift_report.to_dict(), indent=2))

**Test coverage:** 171 unit tests, all passing (1 skipped — rclpy not on Colab). 4 framework abstractions + 7 JD modules + bonus triage agent. Model-agnostic (swap to YOLO27s in one line). Production patterns (thread-safe, error-resilient, import-safe).

**Next steps in the project:**
- Real TensorRT export on T4 / Jetson (use `scripts/export_tensorrt.py`)
- Mavis `conveyor-perception-coach` custom agent (created)
- Docker Compose stack with ROS 2 + RViz (created — `docker/docker-compose.yml`)
- Full 30-epoch training on Colab T4 (mAP50 likely 0.75+)

See `docs/COLAB_60SEC.md` for the 1-cell fast path. See `docs/LIVE_DEMO_CHECKLIST.md` for the pre-call 10-min checklist.
